## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [35]:
from dotenv import load_dotenv

# AsyncOpenAI and OpenAIChatCompletionsModel are used to interact with OpenAI-compatible models asynchronously.
from agents import Agent, Runner, AsyncOpenAI, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from openai.types.responses import ResponseTextDeltaEvent

from typing import Dict
import sendgrid
import os

from pydantic import BaseModel

import sys
sys.path.append('../my-utils')
from my_mailer import send_email as mailer_send_email
from my_mailer import send_html_email as mailer_send_html_email

In [36]:
load_dotenv(override=True)

True

In [37]:
grok_api_key = os.getenv('GROK_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
ollama_api_key = os.getenv('OLLAMA_API_KEY') # can be anything, it really doesn't need one, but the api needs it to run

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:4]}")
else:
    print("Google API Key not set (and this is optional)")

if ollama_api_key:
    print(f"Ollama API Key exists and begins {ollama_api_key[:4]}")
else:
    print("Ollama API Key not set (and this is optional)")
    ollama_api_key = "ollama"

Grok API Key exists and begins xai-
Google API Key exists and begins AIza
Ollama API Key exists and begins olla


In [38]:
# 3 different sales agent personas

# 1 - Professional and Serious
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

# 2 - Humorous and Engaging
instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

# 3 - Busy and Concise
instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

### It's easy to use any models with OpenAI compatible endpoints

In [39]:
GEMINI_BASE_URL = os.getenv('GEMINI_BASE_URL')
GROK_BASE_URL = os.getenv('GROK_BASE_URL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')

print(f"GEMINI_BASE_URL: {GEMINI_BASE_URL}")
print(f"GROK_BASE_URL: {GROK_BASE_URL}")
print(f"OLLAMA_BASE_URL: {OLLAMA_BASE_URL}")

GEMINI_BASE_URL: https://generativelanguage.googleapis.com/v1beta/openai/
GROK_BASE_URL: https://api.x.ai/v1
OLLAMA_BASE_URL: http://localhost:11434/v1


In [40]:

## Step 1 Set up the clients, just like the OpenAI ones but using AsyncOpenAI instead
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key=ollama_api_key)

## Step 2 Set up the models
grok_model = OpenAIChatCompletionsModel(model="grok-4-fast", openai_client=grok_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.5-flash", openai_client=gemini_client)
ollama_gemma_model = OpenAIChatCompletionsModel(model=os.getenv("GEMMA_MODEL"), openai_client=ollama_client)
ollama_deepseek_model = OpenAIChatCompletionsModel(model=os.getenv("DEEPSEEK_MODEL"), openai_client=ollama_client)
ollama_qwen_model = OpenAIChatCompletionsModel(model=os.getenv("QWEN_MODEL"), openai_client=ollama_client)

In [41]:
# Now when you set up the agents, you use the models setup above, if just using openai you would do model="gpt-4" or similar, but it also takes an OpenAIChatCompletionsModel instance
sales_agent1 = Agent(name="GrokFast Sales Agent", instructions=instructions1, model=grok_model)
sales_agent2 =  Agent(name="GrokFast Sales Agent", instructions=instructions2, model=grok_model)
sales_agent3  = Agent(name="GrokFast.3 Sales Agent",instructions=instructions3,model=grok_model)

In [42]:
# result = Runner.run_streamed(sales_agent1, input="Write a cold sales email, only include the email body, no comments on the process.")
# async for event in result.stream_events():
#     if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
#         print(event.data.delta, end="", flush=True)

In [43]:
# result2 = Runner.run_streamed(sales_agent2, input="Write a cold sales email, only include the email body, no comments on the process.")
# async for event in result2.stream_events():
#     if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
#         print(event.data.delta, end="", flush=True)

In [44]:
# result3 = Runner.run_streamed(sales_agent3, input="Write a cold sales email, only include the email body, no comments on the process.")
# async for event in result3.stream_events():
#     if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
#         print(event.data.delta, end="", flush=True)

In [45]:
description = "Write a cold sales email"

# tools for each agent converting the agent to a tool using the as_tool method
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [46]:
# convert the send email function to a tool using the function_tool decorator

@function_tool
async def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    await mailer_send_html_email(
        to="3liotm@gmail.com",  # Your email
        subject=subject,
        html_body=html_body
    )
    return {"status": "success"}

In [47]:
# Additional tools for subject writing and HTML conversion, for a seperate agent to use, this will be the handoff agent
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

# Create the subject writer and html converter agents using the gemini_model
# convert them to tools using the as_tool method
subject_writer = Agent(name="Email subject writer gemini-fast", instructions=subject_instructions, model=gemini_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter gemini-fast", instructions=html_instructions, model=gemini_model)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [48]:
# Combine all email related tools into a list
email_tools = [subject_tool, html_tool, send_html_email]

In [49]:
# instructions for the emailer agent
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

# create the emailer agent using the grok_model
emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model=grok_model,
    handoff_description="Convert an email to HTML and send it")

In [50]:
# Create the main sales agent that uses the 3 sales agents as tools and hands off to the emailer agent
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [51]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.  However you can only use them a total of 3 times each.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=grok_model)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

HTML email sent successfully to 3liotm@gmail.com


## Check out the trace:

https://platform.openai.com/traces

In [53]:
# the pydantic model for the guardrail output
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

# guardrail_agent setup
guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model=gemini_model
)

In [54]:
# input guardrail function to check for personal names in the message
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [56]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model=grok_model,
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Alice"

# this is setup to fail so it will look like an error with the cell erroring
# this is becuase the message = above includes a personal name "Alice" which the guardrail will catch
# the error message will indicate the GuardrailFunctionOutput triggered the tripwire
with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

## Check out the trace:

https://platform.openai.com/traces

In [57]:

# this one should work as there is no personal name in the message
message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

HTML email sent successfully to 3liotm@gmail.com


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>